# Day 5 — Introduction to PyTorch & Tensors

**Agri-AI Bootcamp** · Phase 2 of 5: *Deep Learning & PyTorch* · Days 5–8

> Port the Day 2 regression to PyTorch with torch.tensor, .to('cuda') and loss.backward().

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

A tensor is an array that can live on the GPU and remember how it was computed. Autograd records operations so loss.backward() fills every parameter’s gradient automatically.

### 🌾 Agriculture example

The Day 2 nitrogen model: x and y become tensors on cuda, w and b get requires_grad=True, and the hand-written gradient lines disappear.

> **Watch out:** CPU and CUDA tensors can’t be combined; move model and data together, and call optimizer.zero_grad() each step.

## 2. Setup

In [ ]:
import torch, numpy as np
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, "(Runtime -> Change runtime type -> T4 GPU for cuda)")

## 3. Data

In [ ]:
# ---- Farm-season data --------------------------------------------------------
# Course dataset: Kaggle "Crop Yield Prediction".  Kaggle needs an API token, so
# we simulate 200 farm-seasons with the same columns.  Replace this cell with
# pd.read_csv("yield_df.csv") once you have downloaded the real file.
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 200
year      = rng.integers(2005, 2025, n)
rainfall  = rng.normal(850, 180, n).clip(300, 1500)      # mm / season
fertiliser= rng.normal(120,  35, n).clip(20, 260)        # kg N / ha
# "true" relationship + noise, plus one drought year that behaves differently
yield_t   = (0.9 + 0.0022 * rainfall + 0.020 * fertiliser
             - 0.0000012 * rainfall ** 2 + rng.normal(0, 0.45, n))
yield_t[year == 2012] -= 1.8                             # drought season
df = pd.DataFrame(dict(year=year, rainfall=rainfall,
                       fertiliser=fertiliser, yield_t_ha=yield_t.clip(0.2, None)))
X = df[["rainfall", "fertiliser", "year"]].to_numpy(float)
y = df["yield_t_ha"].to_numpy(float)
df.head()

x_np = (df.fertiliser.to_numpy(float) - df.fertiliser.mean()) / df.fertiliser.std()
y_np = df.yield_t_ha.to_numpy(float)
x_t = torch.tensor(x_np, dtype=torch.float32, device=dev)
y_t = torch.tensor(y_np, dtype=torch.float32, device=dev)
print(x_t.shape, x_t.dtype, x_t.device)

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
import torch
dev="cuda" if torch.cuda.is_available() else "cpu"
x=torch.tensor(x,dtype=torch.float32,device=dev)
y=torch.tensor(y,dtype=torch.float32,device=dev)
w=torch.zeros(1,device=dev,requires_grad=True)
b=torch.zeros(1,device=dev,requires_grad=True)
loss=((w*x+b-y)**2).mean(); loss.backward()
```

## 5. Worked solution

In [ ]:
# ---- Tensors 101 -------------------------------------------------------------
a = torch.arange(6.).reshape(2, 3)
print(a, "\nshape", tuple(a.shape), "| sum", a.sum().item(), "| mean dim0", a.mean(0))
print("to numpy:", a.numpy().__class__.__name__, "| from numpy:",
      torch.from_numpy(np.ones(3, dtype=np.float32)))

# ---- Autograd replaces the hand-written gradients of Day 2 -------------------
w = torch.zeros(1, device=dev, requires_grad=True)
b = torch.zeros(1, device=dev, requires_grad=True)
lr = 0.05
for epoch in range(500):
    loss = ((w * x_t + b - y_t) ** 2).mean()
    loss.backward()                      # fills w.grad and b.grad
    with torch.no_grad():                # update outside the graph
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()       # gradients ACCUMULATE -- always clear
    if epoch % 100 == 0:
        print(f"epoch {epoch:>3}  loss {loss.item():.4f}  w {w.item():.3f}  b {b.item():.3f}")
print(f"final w={w.item():.3f} b={b.item():.3f}")

# ---- The same thing with nn + optim (what you will use from now on) ----------
import torch.nn as nn
model = nn.Linear(1, 1).to(dev)
opt   = torch.optim.SGD(model.parameters(), lr=0.05)
lossf = nn.MSELoss()
X1 = x_t.unsqueeze(1); Y1 = y_t.unsqueeze(1)
for epoch in range(500):
    opt.zero_grad()                      # <- the zero_grad() habit
    l = lossf(model(X1), Y1)
    l.backward(); opt.step()
print(f"nn.Linear -> w={model.weight.item():.3f} b={model.bias.item():.3f} loss {l.item():.4f}")

# ---- The classic shape/device mistakes --------------------------------------
if dev == "cuda":
    try:
        torch.ones(3) + torch.ones(3, device=dev)
    except RuntimeError as e:
        print("\ndevice mismatch ->", str(e)[:90])
else:
    print("\ndevice mismatch -> (needs a GPU runtime to trigger: CPU + CUDA tensors cannot mix)")
try:
    nn.Linear(1, 1)(x_t.cpu())           # 1-D input into a Linear layer
except RuntimeError as e:
    print("shape mistake  ->", str(e)[:90])

## 6. Your turn

1. Replace SGD with Adam and compare how many epochs you need for the same loss.
2. Move a tensor between CPU and GPU with `.to()` and time 500 epochs on each.
3. Set `requires_grad=False` on w and confirm the update stops.

In [ ]:
# TODO 1: Adam vs SGD
import torch.nn as nn, time
for name, make in [("SGD", lambda p: torch.optim.SGD(p, lr=.05)),
                   ("Adam", lambda p: torch.optim.Adam(p, lr=.05))]:
    mdl = nn.Linear(1, 1).to(dev); op = make(mdl.parameters()); f = nn.MSELoss()
    t0 = time.time()
    for _ in range(500):
        op.zero_grad(); l = f(mdl(x_t.unsqueeze(1)), y_t.unsqueeze(1)); l.backward(); op.step()
    print(f"{name:<5} loss {l.item():.4f}  {time.time()-t0:.2f}s")

# TODO 2: time CPU vs GPU
# TODO 3: requires_grad=False experiment

## 7. Dataset and references

**Dataset:** [Kaggle: Crop Yield Prediction](https://www.kaggle.com/datasets/patelris/crop-yield-prediction-dataset)

**References**

- [Learning PyTorch with Examples](https://pytorch.org/tutorials/beginner/pytorch_with_examples.html)

**Course material:** PyTorch: Learning PyTorch with Examples

---
**Next:** Day 6 — Multi-Layer Perceptrons (MLPs)